# audio

> Route handlers for alignment audio playback controls

In [ ]:
#| default_exp routes.audio

In [ ]:
#| export
from typing import Dict, Callable, Tuple

from fasthtml.common import Script
from cjm_fasthtml_app_core.core.routing import APIRouter

from cjm_fasthtml_interactions.core.state_store import get_session_id

# Web Audio library — shared speed-change JS generator
from cjm_fasthtml_web_audio.js import generate_speed_change_js

from cjm_transcript_vad_align.models import AlignmentUrls
from cjm_transcript_vad_align.routes.core import (
    WorkflowStateStore, _update_alignment_state
)
from cjm_transcript_vad_align.components.callbacks import ALIGN_AUDIO_CONFIG

## Auto-Navigate Toggle Handler

Updates auto-navigate state and returns JavaScript to update the client-side flag.

In [ ]:
#| export
def _generate_auto_nav_js(
    enabled:bool  # Whether auto-navigate is enabled
) -> str:  # JavaScript to update auto-navigate flag
    """Generate JS to update auto-navigate flag via shared library."""
    enabled_str = "true" if enabled else "false"
    return f"""
        if (window.setAlignAutoNavigate) window.setAlignAutoNavigate({enabled_str});
    """

## Router Initialization

Creates audio control routes and returns the router with route functions.

In [ ]:
#| export
def init_audio_router(
    state_store:WorkflowStateStore,  # The workflow state store
    workflow_id:str,  # The workflow identifier
    prefix:str,  # Base prefix for audio routes
    urls:AlignmentUrls,  # URL bundle to populate
) -> Tuple[APIRouter, Dict[str, Callable]]:  # (router, routes dict)
    """Initialize alignment audio control routes."""
    router = APIRouter(prefix=prefix)
    routes = {}

    @router.post("/toggle_auto_nav")
    async def toggle_auto_nav(request, sess):
        """Handle auto-navigate toggle."""
        session_id = get_session_id(sess)
        form_data = await request.form()
        auto_navigate = "auto_navigate" in form_data

        _update_alignment_state(
            state_store, workflow_id, session_id,
            auto_navigate=auto_navigate
        )

        return Script(_generate_auto_nav_js(auto_navigate))

    routes["toggle_auto_nav"] = toggle_auto_nav

    @router.post("/speed_change")
    def speed_change(request, sess, speed:float):
        """Handle playback speed change."""
        session_id = get_session_id(sess)

        _update_alignment_state(
            state_store, workflow_id, session_id,
            playback_speed=speed,
        )

        return Script(generate_speed_change_js(ALIGN_AUDIO_CONFIG, speed))

    routes["speed_change"] = speed_change

    return router, routes

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()